In [ ]:
UPDATE = False
DROP = False
PresentDate = "2021-12-21"

In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections
import matplotlib.pyplot as plt
# %matplotlib inline

import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

In [ ]:
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/df_dates.pkl', compression=None)

# Preparation of Data Frames

In [ ]:
df  = pd.DataFrame()

## Sentiment

In [ ]:
# Reading the saved data pickle file

df_sentiment = pd.DataFrame()

df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment.pkl', compression=None)

# df_sentiment.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
df_sentiment.drop(columns=["Company"], inplace=True)
df_sentiment.index = pd.to_datetime(df_sentiment.index)
df_sentiment.Date = pd.to_datetime(df_sentiment.Date)
df_sentiment = df_sentiment[df_sentiment["PressReleaseCount"]>0]
df_sentiment = df_sentiment[df_sentiment["Sentiment"]>0]

df_sentiment = df_sentiment.ffill()

df_sentiment["Sentiment(t-1)"] = df_sentiment["Sentiment"].shift(periods=1)
df_sentiment["Sentiment(t-2)"] = df_sentiment["Sentiment"].shift(periods=2)
df_sentiment["Sentiment(t-1)"] = df_sentiment["Sentiment(t-1)"].ffill()
df_sentiment["Sentiment(t-2)"] = df_sentiment["Sentiment(t-2)"].ffill()

# SENTIMENT RETURN
# df_sentiment["Sentiment_return"]=(df_sentiment["Sentiment(t-1)"]-df_sentiment["Sentiment(t-2)"])/df_sentiment["Sentiment(t-2)"]
# SENTIMENT LN RETURN
df_sentiment["Sentiment_ln_return"]=np.log(df_sentiment["Sentiment"]/df_sentiment["Sentiment(t-1)"])
df_sentiment["Sentiment_ln_return(t-1)"]=np.log(df_sentiment["Sentiment(t-1)"]/df_sentiment["Sentiment(t-2)"])

# SENTIMENT EWM
df_sentiment["Sentiment_ewm10"]=df_sentiment["Sentiment"].ewm(10).mean()
df_sentiment["Sentiment_ewm20"]=df_sentiment["Sentiment"].ewm(20).mean()

# POSITIVE LN RETURN
df_sentiment["pos(t-1)"] = df_sentiment["pos"].shift(periods=1)
df_sentiment["pos(t-2)"] = df_sentiment["pos"].shift(periods=2)
df_sentiment["pos(t-1)"] = df_sentiment["pos(t-1)"].ffill()
df_sentiment["pos(t-2)"] = df_sentiment["pos(t-2)"].ffill()
df_sentiment["pos_ln_return"]=np.log(df_sentiment["pos"]/df_sentiment["pos(t-1)"])
df_sentiment["pos_ln_return(t-1)"]=np.log(df_sentiment["pos(t-1)"]/df_sentiment["pos(t-2)"])

# NEGATIVE LN RETURN
df_sentiment["neg(t-1)"] = df_sentiment["neg"].shift(periods=1)
df_sentiment["neg(t-2)"] = df_sentiment["neg"].shift(periods=2)
df_sentiment["neg(t-1)"] = df_sentiment["neg(t-1)"].ffill()
df_sentiment["neg(t-2)"] = df_sentiment["neg(t-2)"].ffill()
df_sentiment["neg_ln_return"]=np.log(df_sentiment["neg"]/df_sentiment["neg(t-1)"])
df_sentiment["neg_ln_return(t-1)"]=np.log(df_sentiment["neg(t-1)"]/df_sentiment["neg(t-2)"])

# NEUTRAL LN RETURN
df_sentiment["neu(t-1)"] = df_sentiment["neu"].shift(periods=1)
df_sentiment["neu(t-2)"] = df_sentiment["neu"].shift(periods=2)
df_sentiment["neu(t-1)"] = df_sentiment["neu(t-1)"].ffill()
df_sentiment["neu(t-2)"] = df_sentiment["neu(t-2)"].ffill()
df_sentiment["neu_ln_return"]=np.log(df_sentiment["neu"]/df_sentiment["neu(t-1)"])
df_sentiment["neu_ln_return(t-1)"]=np.log(df_sentiment["neu(t-1)"]/df_sentiment["neu(t-2)"])

# COMPOUND LN RETURN
df_sentiment["compound(t-1)"] = df_sentiment["compound"].shift(periods=1)
df_sentiment["compound(t-2)"] = df_sentiment["compound"].shift(periods=2)
df_sentiment["compound(t-1)"] = df_sentiment["compound(t-1)"].ffill()
df_sentiment["compound(t-2)"] = df_sentiment["compound(t-2)"].ffill()
df_sentiment["compound_ln_return"]=np.log(df_sentiment["compound"]/df_sentiment["compound(t-1)"])
df_sentiment["compound_ln_return(t-1)"]=np.log(df_sentiment["compound(t-1)"]/df_sentiment["compound(t-2)"])

df_sentiment["polarity(t-1)"] = df_sentiment["polarity"].shift(periods=1)
df_sentiment["subjectivity(t-1)"] = df_sentiment["subjectivity"].shift(periods=1)

# PRESS RELEASE COUNT
df_sentiment["PressReleaseCount_ewm10"] = df_sentiment["PressReleaseCount"].ewm(10).mean()

df_sentiment["PressReleaseCount(t-1)"] = df_sentiment["PressReleaseCount"].shift(periods=1)
df_sentiment["PressReleaseCount(t-2)"] = df_sentiment["PressReleaseCount"].shift(periods=2)
df_sentiment["PressReleaseCount(t-1)"] = df_sentiment["PressReleaseCount(t-1)"].ffill()
df_sentiment["PressReleaseCount(t-2)"] = df_sentiment["PressReleaseCount(t-2)"].ffill()
df_sentiment["PressReleaseCount_ln_return"]=np.log(df_sentiment["PressReleaseCount"]/df_sentiment["PressReleaseCount(t-1)"])

df_sentiment['WordCount'] = df_sentiment['Articles'].apply(lambda x:len(str(x).split())) #Number Of words in main text
df_sentiment["WordCount(t-1)"] = df_sentiment["WordCount"].shift(periods=1)
df_sentiment["WordCount(t-2)"] = df_sentiment["WordCount"].shift(periods=2)
df_sentiment["WordCount(t-1)"] = df_sentiment["WordCount(t-1)"].ffill()
df_sentiment["WordCount(t-2)"] = df_sentiment["WordCount(t-2)"].ffill()
df_sentiment["WordCount_ln_return"]=np.log(df_sentiment["WordCount"]/df_sentiment["WordCount(t-1)"])

## VIX

In [ ]:
if (UPDATE):
    from yahoo_fin.stock_info import get_data
    df_vix = get_data("^VIX", start_date="25/09/2019", end_date=PresentDate, index_as_date = True, interval="1d")
    df_vix.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl")

In [ ]:
df_vix=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl', compression=None)

df_vix.index = pd.to_datetime(df_vix.index)
df_vix["Date"]= pd.to_datetime(df_vix.index)

df_vix["vix"] = df_vix["adjclose"]
if (DROP==True):
    df_vix.drop(columns=["high", "low","close", "adjclose", "ticker", "volume"], inplace=True)

df_vix["vix(t-1)"] = df_vix["vix"].shift(periods=1)
df_vix["vix(t-2)"] = df_vix["vix"].shift(periods=2)
df_vix["vix_ln_return"]=np.log(df_vix["vix"]/df_vix["vix(t-1)"])
df_vix["vix_ln_return(t-1)"]=np.log(df_vix["vix(t-1)"]/df_vix["vix(t-2)"])
df_vix["vix_ewm"]=df_vix["vix"].ewm(10).mean()

## S&P500

In [ ]:
if (UPDATE):
    from yahoo_fin.stock_info import get_data
    df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date=PresentDate, index_as_date = True, interval="1d")
    df_sp500.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl")

In [ ]:
df_sp500=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl', compression=None)

df_sp500.index = pd.to_datetime(df_sp500.index)
df_sp500["Date"]= pd.to_datetime(df_sp500.index)

df_sp500["sp500"] = df_sp500["adjclose"]
if (DROP==True):
    df_sp500.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_sp500["sp500"]  = df_sp500["sp500"].ffill()

df_sp500["sp500(t-1)"] = df_sp500["sp500"].shift(periods=1)
df_sp500["sp500(t-2)"] = df_sp500["sp500"].shift(periods=2)

# sp500 RETURN
df_sp500["sp500_return"]=(df_sp500["sp500"]-df_sp500["sp500(t-1)"])/df_sp500["sp500(t-1)"]
# sp500 LN RETURN
df_sp500["sp500_ln_return"]=np.log(df_sp500["sp500"]/df_sp500["sp500(t-1)"])

df_sp500["sp500_ln_return(t-1)"]=np.log(df_sp500["sp500(t-1)"]/df_sp500["sp500(t-2)"])

# sp500 EWM
df_sp500["sp500_ewm"]=df_sp500["sp500"].ewm(10).mean()

# MERGING 

In [ ]:
df = pd.merge(df_sentiment,df_vix, on='Date')
df = df.ffill()
df = df.sort_values(by="Date")

df = pd.merge(df, df_sp500, on='Date')#,how='outer'
df = df.ffill()
df = df.sort_values(by="Date")

# DAY MONTH YEAR

In [ ]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['year'] = [d.year for d in df.Date]
df['month'] = [d.strftime('%b') for d in df.Date]
df['day'] = [d.day_name() for d in df.Date]
years = df['year'].unique()
df['month-year'] = [d.strftime('%b') + str(d.year) for d in df.Date]
df['quarter'] = pd.PeriodIndex(df.Date, freq='Q')

# THREE PERIODS

In [ ]:
df = df.ffill()
df.index = df.Date
df.ffill(inplace=True)

In [ ]:
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':PresentDate]
df_sentiment = df_sentiment.loc['2020-06-01':PresentDate]
df_vix = df_vix.loc['2020-06-01':PresentDate]

In [ ]:
mx = df_part3.loc[df_part3['Sentiment_ln_return'] != np.inf, 'Sentiment_ln_return'].max()
df_part3['Sentiment_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['Sentiment_ln_return'] != -np.inf, 'Sentiment_ln_return'].min()
df_part3['Sentiment_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['neg_ln_return'] != np.inf, 'neg_ln_return'].max()
df_part3['neg_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['neg_ln_return'] != -np.inf, 'neg_ln_return'].min()
df_part3['neg_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['neu_ln_return'] != np.inf, 'neu_ln_return'].max()
df_part3['neu_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['neu_ln_return'] != -np.inf, 'neu_ln_return'].min()
df_part3['neu_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['compound_ln_return'] != np.inf, 'compound_ln_return'].max()
df_part3['compound_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['compound_ln_return'] != -np.inf, 'compound_ln_return'].min()
df_part3['compound_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['pos_ln_return'] != np.inf, 'pos_ln_return'].max()
df_part3['pos_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['pos_ln_return'] != -np.inf, 'pos_ln_return'].min()
df_part3['pos_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['neg_ln_return'] != np.inf, 'neg_ln_return'].max()
df_part3['neg_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['neg_ln_return'] != -np.inf, 'neg_ln_return'].min()
df_part3['neg_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df_part3.loc[df_part3['neu_ln_return'] != np.inf, 'neu_ln_return'].max()
df_part3['neu_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['neu_ln_return'] != -np.inf, 'neu_ln_return'].min()
df_part3['neu_ln_return'].replace(-np.inf,mn,inplace=True)

In [ ]:
mn = df_part3.loc[df_part3['Sentiment_ln_return'] != -np.inf, 'Sentiment_ln_return'].min()
df_part3['Sentiment_ln_return'] = df_part3['Sentiment_ln_return'].fillna(mn)

mn = df_part3.loc[df_part3['vix_ln_return'] != -np.inf, 'vix_ln_return'].min()
df_part3['vix_ln_return'] = df_part3['vix_ln_return'].fillna(mn)

mn = df_part3.loc[df_part3['sp500_ln_return'] != -np.inf, 'sp500_ln_return'].min()
df_part3['sp500_ln_return'] = df_part3['sp500_ln_return'].fillna(mn)

mn = df_part3.loc[df_part3['PressReleaseCount_ln_return'] != -np.inf, 'PressReleaseCount_ln_return'].min()
df_part3['PressReleaseCount_ln_return'] = df_part3['PressReleaseCount_ln_return'].fillna(mn)

In [ ]:
# df = pd.DataFrame()
# for i in (range(24)):

#     # df["Sentiment Natural Log"].ffill(inplace=True)
#     df["Sentiment Natural Log"] = df.loc[df["Sentiment Natural Log"]!=-np.inf]
#     df["Sentiment Natural Log"] = df.loc[df["Sentiment Natural Log"]!=np.inf]

#     df["Sentiment Natural Log"] = df["Sentiment Natural Log"].replace(np.inf,0)

#     df["Sentiment Natural Log"] = df["Sentiment Natural Log"].replace(-np.inf,0)

In [ ]:
# df_part3.set_index(['Date', 'Year', 'Month', 'Day'], inplace=True)

In [ ]:
df_part3.loc[df_part3["PressReleaseCount"]>0]

In [ ]:
df_part3['Articles'].apply(lambda x:len(str(x).split())).sum() #Number Of words in main text

In [ ]:
df_part3[["vix_ln_return","Sentiment_ln_return","Sentiment"]].sort_values(by="vix_ln_return")

# General Inquirer

In [ ]:
df_gi=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentGeneralInquirer.pkl', compression=None)

# df_gi.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
df_gi.drop(columns=["Company"], inplace=True)
df_gi.Date = pd.to_datetime(df_gi.Date)
df_gi.index = pd.to_datetime(df_gi.Date)
df_gi = df_gi[df_gi["PressReleaseCount"]>0]
df_gi = df_gi[df_gi["Sentiment"]>0]

df_gi = df_gi.ffill()


df_gi.rename(columns={"Sentiment":"GI_Sentiment",
                        "PressReleaseCount":"GI_PressReleaseCount" ,
                        "Positive":"GI_Positive",
                        "Negative":"GI_Negative", 
                        "Polarity":"GI_Polarity", 
                        "Subjectivity":"GI_Subjectivity"}
                        ,inplace=True)

df_gi["GI_Sentiment(t-1)"] = df_gi["GI_Sentiment"].shift(periods=1)
df_gi["GI_Sentiment(t-2)"] = df_gi["GI_Sentiment"].shift(periods=2)
df_gi["GI_Sentiment(t-1)"] = df_gi["GI_Sentiment(t-1)"].ffill()
df_gi["GI_Sentiment(t-2)"] = df_gi["GI_Sentiment(t-2)"].ffill()

# SENTIMENT RETURN
# df_gi["GI_Sentiment_return"]=(df_gi["GI_Sentiment(t-1)"]-df_gi["GI_Sentiment(t-2)"])/df_gi["GI_Sentiment(t-2)"]
# SENTIMENT LN RETURN
df_gi["GI_Sentiment_ln_return"]=np.log(df_gi["GI_Sentiment"]/df_gi["GI_Sentiment(t-1)"])
df_gi["GI_Sentiment_ln_return(t-1)"]=np.log(df_gi["GI_Sentiment(t-1)"]/df_gi["GI_Sentiment(t-2)"])

# SENTIMENT EWM
df_gi["GI_Sentiment_ewm10"]=df_gi["GI_Sentiment"].ewm(10).mean()
df_gi["GI_Sentiment_ewm20"]=df_gi["GI_Sentiment"].ewm(20).mean()

# POSITIVE LN RETURN
df_gi["GI_Positive(t-1)"] = df_gi["GI_Positive"].shift(periods=1)
df_gi["GI_Positive(t-2)"] = df_gi["GI_Positive"].shift(periods=2)
df_gi["GI_Positive(t-1)"] = df_gi["GI_Positive(t-1)"].ffill()
df_gi["GI_Positive(t-2)"] = df_gi["GI_Positive(t-2)"].ffill()
df_gi["GI_Positive_ln_return"]=np.log(df_gi["GI_Positive"]/df_gi["GI_Positive(t-1)"])

# NEGATIVE LN RETURN
df_gi["GI_Negative(t-1)"] = df_gi["GI_Negative"].shift(periods=1)
df_gi["GI_Negative(t-2)"] = df_gi["GI_Negative"].shift(periods=2)
df_gi["GI_Negative(t-1)"] = df_gi["GI_Negative(t-1)"].ffill()
df_gi["GI_Negative(t-2)"] = df_gi["GI_Negative(t-2)"].ffill()
df_gi["GI_Negative_ln_return"]=np.log(df_gi["GI_Negative"]/df_gi["GI_Negative(t-1)"])

df_gi["GI_Polarity(t-1)"] = df_gi["GI_Polarity"].shift(periods=1)
df_gi["GI_Subjectivity(t-1)"] = df_gi["GI_Subjectivity"].shift(periods=1)

# # PRESS RELEASE COUNT
# df_gi["GI_PressReleaseCount_ewm10"] = df_gi["PressReleaseCount"].ewm(10).mean()

# df_gi["GI_PressReleaseCount(t-1)"] = df_gi["PressReleaseCount"].shift(periods=1)
# df_gi["GI_PressReleaseCount(t-2)"] = df_gi["PressReleaseCount"].shift(periods=2)
# df_gi["GI_PressReleaseCount(t-1)"] = df_gi["GI_PressReleaseCount(t-1)"].ffill()
# df_gi["GI_PressReleaseCount(t-2)"] = df_gi["GI_PressReleaseCount(t-2)"].ffill()
# df_gi["GI_PressReleaseCount_ln_return"]=np.log(df_gi["GI_PressReleaseCount"]/df_gi["GI_PressReleaseCount(t-1)"])

# df_gi['GI_WordCount'] = df_gi['Articles'].apply(lambda x:len(str(x).split())) #Number Of words in main text
# df_gi["GI_WordCount(t-1)"] = df_gi["GI_WordCount"].shift(periods=1)
# df_gi["GI_WordCount(t-2)"] = df_gi["GI_WordCount"].shift(periods=2)
# df_gi["GI_WordCount(t-1)"] = df_gi["GI_WordCount(t-1)"].ffill()
# df_gi["GI_WordCount(t-2)"] = df_gi["GI_WordCount(t-2)"].ffill()
# df_gi["GI_WordCount_ln_return"]=np.log(df_gi["GI_WordCount"]/df_gi["GI_WordCount(t-1)"])

df_gi = df_gi.loc["2020-05-31":"2021-12-31"]

In [ ]:
df_gi.columns

In [ ]:
df_gi.drop(columns=["Date",'Adjectives', 'Verbs', 'Adverbs', 'Nouns', "Articles", "Title", "Text", 'GI_PressReleaseCount', #'PressReleaseCount(t-1)',
       "MarketCapFactor", "GI_PressReleaseCount",
       #'GI_PressReleaseCount(t-2)', 'GI_PressReleaseCount_ln_return',
       # 'GI_WordCount', 
       #'GI_WordCount(t-1)', 'GI_WordCount(t-2)',
       # 'GI_WordCount_ln_return'
       ], inplace=True)

In [ ]:
df_gi.columns

In [ ]:
df_part3.drop(columns={"Date"},inplace=True)#(inplace=True)

In [ ]:
df = pd.merge(df_part3,df_gi, on='Date')

In [ ]:
df["Date"] = df.index

In [ ]:
df.rename(columns={"vix":"VIX",
                "vix(t-1)":"VIX(t-1)",
                "vix_ln_return":"VIX Natural Log",
                "sp500":"S&P500",
                "sp500(t-1)":"S&P500(t-1)",
                "sp500(t-2)":"S&P500(t-2)",
                "sp500_ln_return":"S&P500 Natural Log",
                "Sentiment_ln_return":"Sentiment Natural Log",
                "Sentiment_ln_return(t-1)":"Sentiment Natural Log(t-1)",
                "pos":"Positive",
                "pos(t-1)":"Positive(t-1)",
                "pos(t-2)":"Positive(t-2)",
                "pos_ln_return":"Positive Natural Log",
                "pos_ln_return(t-1)":"Positive Natural Log(t-1)",
                "neg":"Negative",
                "neg(t-1)":"Negative(t-1)", 
                "neg(t-2)":"Negative(t-2)",
                "neg_ln_return":"Negative Natural Log", 
                "neg_ln_return(t-1)":"Negative Natural Log(t-1)", 
                "neu":"Neutral", 
                "neu(t-1)":"Neutral(t-1)",
                "neu(t-2)":"Neutral(t-2)",
                "neu_ln_return":"Neutral Natural Log",
                "neu_ln_return(t-1)":"Neutral Natural Log(t-1)",
                "compound":"Compound",
                "compound(t-1)":"Compound(t-1)",
                "compound(t-2)":"Compound(t-2)",
                "compound_ln_return":"Compound Natural Log",
                "compound_ln_return(t-1)":"Compound Natural Log(t-1)",
                "polarity":"Polarity",
                "polarity(t-1)":"Polarity(t-1)",
                "polarityB":"PolarityB",
                "subjectivity":"Subjectivity",
                "subjectivity(t-1)":"Subjectivity(t-1)",
                "PressReleaseCount_ln_return":"PressReleaseCount Natural Log",
                "PressReleaseCount_ln_return(t-1)":"PressReleaseCount Natural Log(t-1)",
                "WordCount_ln_return":"WordCount Natural Log",
                "WordCount_ln_return(t-1)":"WordCount Natural Log(t-1)",
                "day":"Day of Week",
                "month":"Month",
                "month-year":"Month-Year",
                "quarter":"Quarter",
                "year":"Year",
                "GI_Sentiment_ln_return":"GI_Sentiment Natural Log",
                "GI_Negative_ln_return":"GI_Negative Natural Log",
                "GI_Positive_ln_return":"GI_Positive Natural Log"}
                ,inplace=True)

In [ ]:
df.columns

In [ ]:
df_part1.rename(columns={"vix":"VIX",
                "vix(t-1)":"VIX(t-1)",
                "vix_ln_return":"VIX Natural Log",
                "sp500":"S&P500",
                "sp500(t-1)":"S&P500(t-1)",
                "sp500(t-2)":"S&P500(t-2)",
                "sp500_ln_return":"S&P500 Natural Log" ,
                "pos":"Positive",
                "neg":"Negative", 
                "neu":"Neutral", 
                "pos(t-1)":"Positive(t-1)",
                "pos(t-2)":"Positive(t-2)", 
                "neg(t-1)":"Negative(t-1)", 
                "neg(t-2)":"Negative(t-2)", 
                "neu(t-1)":"Neutral(t-1)",
                "neu(t-2)":"Neutral(t-2)",
                "compound":"Compound",
                "compound(t-1)":"Compound(t-1)",
                "compound(t-2)":"Compound(t-2)",
                "compound_ln_return":"Compound Natural Log",
                "polarity":"Polarity",
                "subjectivity":"Subjectivity",
                "polarity(t-1)":"Polarity(t-1)",
                "subjectivity(t-1)":"Subjectivity(t-1)",
                "Sentiment_ln_return":"Sentiment Natural Log",
                "pos_ln_return":"Positive Natural Log",
                "neg_ln_return":"Negative Natural Log",
                "neu_ln_return":"Neutral Natural Log",
                "PressReleaseCount_ln_return":"PressReleaseCount Natural Log",
                "WordCount_ln_return":"Word Count Ln Return",
                "day":"Day of Week",
                "month":"Month",
                "month-year":"Month-Year",
                "quarter":"Quarter",
                "year":"Year",
                "GI_Sentiment_ln_return":"GI_Sentiment Natural Log",
                "GI_Negative_ln_return":"GI_Negative Natural Log",
                "GI_Positive_ln_return":"GI_Positive Natural Log"}
                ,inplace=True)

In [ ]:
df_part2.rename(columns={"vix":"VIX",
                "vix(t-1)":"VIX(t-1)",
                "vix_ln_return":"VIX Natural Log",
                "sp500":"S&P500",
                "sp500(t-1)":"S&P500(t-1)",
                "sp500(t-2)":"S&P500(t-2)",
                "sp500_ln_return":"S&P500 Natural Log" ,
                "pos":"Positive",
                "neg":"Negative", 
                "neu":"Neutral", 
                "pos(t-1)":"Positive(t-1)",
                "pos(t-2)":"Positive(t-2)", 
                "neg(t-1)":"Negative(t-1)", 
                "neg(t-2)":"Negative(t-2)", 
                "neu(t-1)":"Neutral(t-1)",
                "neu(t-2)":"Neutral(t-2)",
                "compound":"Compound",
                "compound(t-1)":"Compound(t-1)",
                "compound(t-2)":"Compound(t-2)",
                "compound_ln_return":"Compound Natural Log",
                "polarity":"Polarity",
                "polarityB":"PolarityB",
                "subjectivity":"Subjectivity",
                "polarity(t-1)":"Polarity(t-1)",
                "subjectivity(t-1)":"Subjectivity(t-1)",
                "Sentiment_ln_return":"Sentiment Natural Log",
                "pos_ln_return":"Positive Natural Log",
                "neg_ln_return":"Negative Natural Log",
                "neu_ln_return":"Neutral Natural Log",
                "PressReleaseCount_ln_return":"PressReleaseCount Natural Log",
                "WordCount_ln_return":"Word Count Ln Return",
                "day":"Day of Week",
                "month":"Month",
                "month-year":"Month-Year",
                "quarter":"Quarter",
                "year":"Year",
                "GI_Sentiment_ln_return":"GI_Sentiment Natural Log",
                "GI_Negative_ln_return":"GI_Negative Natural Log",
                "GI_Positive_ln_return":"GI_Positive Natural Log"}
                ,inplace=True)

In [ ]:
df["PressReleaseCount"]

In [ ]:
df.columns

In [ ]:
# df_part3[["Sentiment","Sentiment Natural Log","vix","vix Natural Log","PressReleaseCount","PressReleaseCount Natural Log"]].to_csv("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/stata/Sentiment.csv")